# Stage 2 · DoT pretraining

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedzait20003/TableMind/blob/main/src/notebooks/02_pretrain.ipynb)

Loads the memory stage from Drive and trains the full DoT model on the WikiSQL training set. TAPAS-small scores the 1024-token input; whole cells are kept (question and header always) until 256 tokens are used; the kept tokens are decoded and re-tokenized for T5, which generates the answer obtained by executing each example's SQL. Cell scores scale the T5 input embeddings, so TAPAS is trained end to end.

**Requires:** `01_memory.ipynb` · **Next:** `03_evaluation.ipynb`

## Setup

On Colab, pick **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
# Environment setup: clones the repo on Colab, uses the local checkout otherwise
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/mohamedzait20003/TableMind.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/TableMind"):
        !git clone -q {REPO_URL} /content/TableMind
    %cd /content/TableMind
    !pip install -q -r requirements.txt
else:
    # Local run (e.g. from src/notebooks): move up to the repository root
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()))

sys.path.insert(0, os.path.abspath("src"))
print("Working directory:", os.getcwd())

In [ ]:
from utils import load_config, set_seed, resolve_device, get_storage_dir

# DoT setup: tapas-small prunes 1024 -> 256 tokens for a t5-base task model,
# trained on the full WikiSQL training set
CONFIG_PATH = "src/config/default.yaml"

config = load_config(CONFIG_PATH)
device = resolve_device(config["system"]["device"])
set_seed(config["system"]["seed"])

# Google Drive on Colab (you will be asked to authorize), ./storage locally
STORAGE = get_storage_dir(config["storage"]["drive_subdir"], config["storage"]["local_dir"])
CHECKPOINT_DIR = config["system"]["checkpoint_dir"]
OUTPUT_DIR = config["system"]["output_dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Config: {CONFIG_PATH} ({config['experiment']['name']})")
print(f"Device: {device} | storage: {STORAGE}")

## Run settings

In [ ]:
from utils import run_name

# DoT reports the median of 3 runs: train and evaluate with SEED = 42, 43, 44.
# USE_MEMORY = False trains the ablation without memory retrieval (all else equal).
# Use the same two values in 02_pretrain and 03_evaluation.
SEED = 42
USE_MEMORY = True

config["system"]["seed"] = SEED
config["model"]["task_transformer"]["use_memory"] = USE_MEMORY
set_seed(SEED)
RUN = run_name(config)
print("Run:", RUN)

## Load the memory stage

In [ ]:
from core import MemoryEncoder, KeyValueMemoryStore
from utils import download_checkpoint, load_checkpoint

memory_encoder = MemoryEncoder.from_config(config)
memory_store = KeyValueMemoryStore()

memory_path = download_checkpoint(config["storage"]["memory_checkpoint"], STORAGE, CHECKPOINT_DIR)
load_checkpoint(memory_path, memory_encoder, memory_store=memory_store)
print("Memory entries:", memory_store.size(), "| retrieval enabled:", USE_MEMORY)

## Load WikiSQL

The official release (~26 MB) is downloaded once; targets come from executing each query on its table. Multi-row answers are joined with ` | `.

In [ ]:
from torch.utils.data import DataLoader
from utils import WikiSQLDataset, collate_fn

train_dataset = WikiSQLDataset.from_config(config, "train")
eval_dataset = WikiSQLDataset.from_config(config, "eval")

pre_cfg = config["pretraining"]
workers = config["system"]["num_workers"]
train_loader = DataLoader(train_dataset, batch_size=pre_cfg["batch_size"], shuffle=True,
                          collate_fn=collate_fn, num_workers=workers)
eval_loader = DataLoader(eval_dataset, batch_size=pre_cfg["batch_size"],
                         collate_fn=collate_fn, num_workers=workers)

example = train_dataset.data[0]
print("Q:", example.question)
print("A:", example.answer, f"({example.aggregation.name})")
example.table.head()

## Build the DoT model

In [ ]:
from core import DoTModel

model = DoTModel.from_config(config, memory_store, memory_encoder)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Train

After every epoch the weights are saved and uploaded as `<run>_latest.pt`, so a Colab disconnect loses at most one epoch. Set `RESUMABLE = True` to also keep optimizer state (≈3× larger files).

In [ ]:
from core import PretrainTrainer
from utils import save_checkpoint, upload_checkpoint

RESUMABLE = False

trainer = PretrainTrainer(
    model, train_loader, eval_loader,
    device=device,
    num_epochs=pre_cfg["num_epochs"],
    learning_rate=pre_cfg["learning_rate"],
    weight_decay=pre_cfg["weight_decay"],
    warmup_ratio=pre_cfg["warmup_ratio"],
    scheduler_type=pre_cfg["scheduler_type"],
    gradient_clipping=pre_cfg["gradient_clipping"],
    gradient_accumulation_steps=pre_cfg["gradient_accumulation_steps"],
    log_steps=pre_cfg["log_steps"],
)

def save_epoch(trainer):
    path = os.path.join(CHECKPOINT_DIR, f"{RUN}_latest.pt")
    save_checkpoint(path, trainer.model, memory_store=memory_store, trainer=trainer,
                    config=config, include_optimizer=RESUMABLE)
    upload_checkpoint(path, STORAGE)

trainer.train(on_epoch_end=save_epoch)

In [ ]:
trainer.plot_training_curves(os.path.join(OUTPUT_DIR, f"{RUN}_pretrain_curves.png"))

## Save and upload the final model

In [ ]:
final_path = os.path.join(CHECKPOINT_DIR, f"{RUN}_final.pt")
save_checkpoint(final_path, model, memory_store=memory_store, trainer=trainer, config=config)
upload_checkpoint(final_path, STORAGE)